# 04 BM25


**Nombre:** Harry Guajan  
**Fecha:** 13 de Noviembre de 2026



## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [ ]:
# Reutilizado de 02_booleansearch y 03_vectorsearch: corpus Gutenberg,
# tokenización/normalización, títulos y las fórmulas TF-IDF/BM25.
from collections import Counter
import math
import re

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from nltk.corpus import gutenberg
    identificadores = gutenberg.fileids()
    if len(identificadores) != 18:
        raise LookupError("La colección Gutenberg de NLTK no está disponible.")
except LookupError:
    nltk.download("gutenberg", quiet=True)
    from nltk.corpus import gutenberg
    identificadores = gutenberg.fileids()
if len(identificadores) != 18:
    raise RuntimeError(f"Se esperaban 18 documentos y se encontraron {len(identificadores)}.")
documentos = [gutenberg.raw(doc_id) for doc_id in identificadores]
token_pattern = re.compile(r"(?u)\b\w\w+\b")

def tokenizar(texto):
    return token_pattern.findall(texto.lower())

def titulo(doc_id):
    nombres = {
        "austen-emma.txt": "Emma", "austen-persuasion.txt": "Persuasion", "austen-sense.txt": "Sense and Sensibility",
        "bible-kjv.txt": "The King James Bible", "blake-poems.txt": "Poems of William Blake", "bryant-stories.txt": "Stories by William Cullen Bryant",
        "burgess-busterbrown.txt": "The Adventures of Buster Bear", "carroll-alice.txt": "Alice's Adventures in Wonderland",
        "chesterton-ball.txt": "The Ball and the Cross", "chesterton-brown.txt": "The Wisdom of Father Brown", "chesterton-thursday.txt": "The Man Who Was Thursday",
        "edgeworth-parents.txt": "The Parent's Assistant", "melville-moby_dick.txt": "Moby Dick", "milton-paradise.txt": "Paradise Lost",
        "shakespeare-caesar.txt": "The Tragedy of Julius Caesar", "shakespeare-hamlet.txt": "The Tragedy of Hamlet", "shakespeare-macbeth.txt": "The Tragedy of Macbeth", "whitman-leaves.txt": "Leaves of Grass",
    }
    return nombres.get(doc_id, doc_id)

documentos_tokenizados = [tokenizar(texto) for texto in documentos]
vocabulario = sorted({termino for tokens in documentos_tokenizados for termino in tokens})
indice_terminos = {termino: i for i, termino in enumerate(vocabulario)}
matriz_tf = np.zeros((len(documentos), len(vocabulario)), dtype=np.int64)
for fila, tokens in enumerate(documentos_tokenizados):
    for termino, frecuencia in Counter(tokens).items():
        matriz_tf[fila, indice_terminos[termino]] = frecuencia

df = np.count_nonzero(matriz_tf, axis=0)
idf = np.log(len(documentos) / df) + 1
longitudes = np.array([len(tokens) for tokens in documentos_tokenizados])
longitud_media = float(longitudes.mean())
tabla_corpus = pd.DataFrame({
    "Identificador de documento": identificadores,
    "Título": [titulo(doc_id) for doc_id in identificadores],
    "Texto": documentos,
    "tokens": documentos_tokenizados,
})
tabla_estadisticas = pd.DataFrame({"término": vocabulario, "df": df, "idf": idf})
print("Se reutilizaron el corpus y el preprocesamiento de 02_booleansearch/03_vectorsearch.")
print(f"Documentos: {len(documentos)} | Vocabulario: {len(vocabulario)} | Longitud media: {longitud_media:.2f}")
display(tabla_corpus[["Identificador de documento", "Título", "Texto", "tokens"]].head())
display(tabla_estadisticas.sort_values(["df", "término"], ascending=[False, True]).head(15))

# La representación de documentos se conserva dispersa, como pide el enunciado.
vectorizador = TfidfVectorizer(
    tokenizer=lambda tokens: tokens, preprocessor=lambda tokens: tokens,
    token_pattern=None, lowercase=False, use_idf=True, smooth_idf=False,
    sublinear_tf=False, norm=None,
)
matriz_tfidf = vectorizador.fit_transform(documentos_tokenizados)
if not np.array_equal(vectorizador.get_feature_names_out(), np.array(vocabulario)):
    raise AssertionError("El vocabulario disperso no coincide con el vocabulario manual.")

def puntuaciones_bm25(consulta_tokens, k1=1.2, b=0.75):
    puntuaciones = np.zeros(len(documentos), dtype=float)
    consulta_unica = set(consulta_tokens)
    for i, tokens in enumerate(documentos_tokenizados):
        frecuencias = Counter(tokens)
        ajuste_longitud = 1 - b + b * len(tokens) / longitud_media
        for termino in consulta_unica:
            if termino not in indice_terminos:
                continue
            frecuencia_documental = df[indice_terminos[termino]]
            idf_bm25 = math.log((len(documentos) - frecuencia_documental + 0.5) / (frecuencia_documental + 0.5) + 1)
            frecuencia = frecuencias[termino]
            puntuaciones[i] += idf_bm25 * frecuencia * (k1 + 1) / (frecuencia + k1 * ajuste_longitud)
    return puntuaciones


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [ ]:
consultas = {
    "frecuente corta": ("man", "encuentra referencias generales a personas"),
    "frecuente media": ("love time", "busca relaciones entre amor y tiempo"),
    "raro corto": ("whale", "busca referencias explícitas a ballenas"),
    "raro media": ("ship whale ocean", "busca escenas marítimas y navegación"),
    "larga narrativa": ("young man went house", "busca una escena narrativa con sujeto, acción y lugar"),
    "larga literaria": ("king queen night word", "busca lenguaje de poder y ambiente nocturno"),
    "temática": ("church family father", "busca contexto religioso y familiar"),
    "mixta": ("science machine life", "busca una combinación científica y existencial"),
}
def ordenar_resultados(scores):
    positivos = [(identificadores[i], titulo(identificadores[i]), float(scores[i]))
                 for i in range(len(identificadores)) if scores[i] > 0]
    return sorted(positivos, key=lambda fila: (-fila[2], str(fila[0])))[:10]

resultados = {}
for nombre, (texto_consulta, objetivo) in consultas.items():
    tokens_consulta = tokenizar(texto_consulta)
    vector_consulta = vectorizador.transform([tokens_consulta])
    scores_tfidf = (matriz_tfidf @ vector_consulta.T).toarray().ravel()
    norma_consulta = float(np.sqrt(vector_consulta.multiply(vector_consulta).sum()))
    normas_documentos = np.sqrt(matriz_tfidf.multiply(matriz_tfidf).sum(axis=1)).A1
    denominadores = normas_documentos * norma_consulta
    scores_coseno = np.divide(scores_tfidf, denominadores, out=np.zeros_like(scores_tfidf), where=denominadores != 0)
    scores_bm25 = puntuaciones_bm25(tokens_consulta)
    resultados[nombre] = {"texto": texto_consulta, "objetivo": objetivo, "tokens": tokens_consulta,
                         "tfidf": scores_coseno, "bm25": scores_bm25}
    print(f"\n## {nombre}: {texto_consulta}\nObjetivo: {objetivo}")
    print("Términos y df:", [(t, int(df[indice_terminos[t]])) for t in sorted(set(tokens_consulta)) if t in indice_terminos])
    display(pd.DataFrame(ordenar_resultados(scores_coseno), columns=["doc_id", "Título", "Coseno"]))
    display(pd.DataFrame(ordenar_resultados(scores_bm25), columns=["doc_id", "Título", "Score BM25"]))


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [ ]:
def tabla_comparacion(nombre):
    datos = resultados[nombre]
    ranking_tfidf = ordenar_resultados(datos["tfidf"])
    ranking_bm25 = ordenar_resultados(datos["bm25"])
    filas = []
    for posicion in range(max(len(ranking_tfidf), len(ranking_bm25))):
        tfidf = ranking_tfidf[posicion] if posicion < len(ranking_tfidf) else ("", "", np.nan)
        bm25 = ranking_bm25[posicion] if posicion < len(ranking_bm25) else ("", "", np.nan)
        filas.append({"Posición": posicion + 1, "ID y título TF-IDF": f"{tfidf[0]} — {tfidf[1]}" if tfidf[0] else "",
                      "Coseno": tfidf[2], "ID y título BM25": f"{bm25[0]} — {bm25[1]}" if bm25[0] else "",
                      "Score BM25": bm25[2]})
    ids_tfidf = {fila[0] for fila in ranking_tfidf}
    ids_bm25 = {fila[0] for fila in ranking_bm25}
    resumen = {"consulta": nombre, "tamaño TF-IDF": len(ranking_tfidf), "tamaño BM25": len(ranking_bm25),
               "intersección top 10": len(ids_tfidf & ids_bm25),
               "primer TF-IDF": ranking_tfidf[0][0] if ranking_tfidf else "",
               "primer BM25": ranking_bm25[0][0] if ranking_bm25 else ""}
    return pd.DataFrame(filas), resumen

resumenes = []
for nombre in consultas:
    tabla, resumen = tabla_comparacion(nombre)
    resumenes.append(resumen)
    print(f"\n## Comparación: {nombre}")
    display(tabla)
display(pd.DataFrame(resumenes))

consulta_explicada = "raro media"
ranking_a = ordenar_resultados(resultados[consulta_explicada]["tfidf"])
ranking_b = ordenar_resultados(resultados[consulta_explicada]["bm25"])
ids_a = [fila[0] for fila in ranking_a]
ids_b = [fila[0] for fila in ranking_b]
candidatos = [doc_id for doc_id in ids_a if doc_id in ids_b and ids_a.index(doc_id) != ids_b.index(doc_id)]
documentos_explicados = candidatos[:2] if len(candidatos) >= 2 else list(dict.fromkeys(ids_a[:2] + ids_b[:2]))[:2]
filas_aportes = []
for doc_id in documentos_explicados:
    i = identificadores.index(doc_id)
    frecuencias = Counter(documentos_tokenizados[i])
    for termino in sorted(set(resultados[consulta_explicada]["tokens"])):
        if termino not in indice_terminos:
            continue
        f_td = frecuencias[termino]
        df_t = int(df[indice_terminos[termino]])
        ajuste = 1 - 0.75 + 0.75 * len(documentos_tokenizados[i]) / longitud_media
        idf_bm = math.log((len(documentos) - df_t + 0.5) / (df_t + 0.5) + 1)
        aporte = idf_bm * f_td * 2.2 / (f_td + 1.2 * ajuste)
        filas_aportes.append({"doc_id": doc_id, "|d|": len(documentos_tokenizados[i]), "término": termino,
                              "f(t,d)": f_td, "df(t)": df_t, "aporte BM25": aporte,
                              "posición TF-IDF": ids_a.index(doc_id) + 1 if doc_id in ids_a else None,
                              "posición BM25": ids_b.index(doc_id) + 1 if doc_id in ids_b else None})
print(f"\n## Aportes por término para la consulta '{resultados[consulta_explicada]['texto']}'")
display(pd.DataFrame(filas_aportes))


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [ ]:
consulta_parametros = "raro media"
configuraciones = [("Base", 1.2, 0.75), ("Sin ajuste por longitud", 1.2, 0.0), ("Mayor efecto de las repeticiones", 2.0, 0.75)]
ids_referencia = [fila[0] for fila in ordenar_resultados(resultados[consulta_parametros]["bm25"])]
documentos_comparados = ids_referencia[:5]
filas_parametros = []
for nombre, k1, b in configuraciones:
    scores = puntuaciones_bm25(resultados[consulta_parametros]["tokens"], k1=k1, b=b)
    ranking = ordenar_resultados(scores)
    ids = [fila[0] for fila in ranking]
    for doc_id in documentos_comparados:
        i = ids.index(doc_id) + 1 if doc_id in ids else None
        filas_parametros.append({"Ejecución": nombre, "k1": k1, "b": b, "doc_id": doc_id,
                                 "posición": i, "score": scores[identificadores.index(doc_id)]})
tabla_parametros = pd.DataFrame(filas_parametros)
display(tabla_parametros.pivot(index="doc_id", columns="Ejecución", values="posición"))
display(tabla_parametros)
print("Interpretación: b=0 elimina la penalización por longitud; por eso favorece relativamente documentos largos.")
print("Al aumentar k1, las repeticiones conservan más peso antes de saturarse; los términos con mayor f(t,d) cambian más su aporte.")
